In [2]:
###########################################################################
## Basic stuff
###########################################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))

###########################################################################
## Utils
###########################################################################
from timeUtils import timestat
from listUtils import getFlatList
from masterDBGate import masterDBGate
from pandas import isna, notna, Series, DataFrame, concat
from uuid import uuid4

###########################################################################
## DB
###########################################################################
from masterManualEntries import masterManualEntries
from masterArtistNameDB import masterArtistNameDB
from masterArtistMerger import masterArtistMerger
from masterMultiArtistDB import masterMultiArtistDB
from masterArtistNameCorrection import masterArtistNameCorrection
from convertByteString import convertByteString
from mainDB import mainDB

In [4]:
from masterDBGate import masterDBGate
mdbGate = masterDBGate()
masterParams = {"AvailableDBs": {db: True for db in mdbGate.getDBs()}}
from fileIO import fileIO
io = fileIO()
io.save(idata=masterParams, ifile="masterParams.yaml")

{'AvailableDBs': {'Discogs': True,
  'AllMusic': True,
  'MusicBrainz': True,
  'RateYourMusic': True,
  'Deezer': True,
  'DeezerAPI': True,
  'LastFM': True,
  'LastFMAPI': True,
  'Genius': True,
  'AlbumOfTheYear': True,
  'KWorbiTunes': True,
  'KWorbSpotify': True,
  'SpotifyCharts': True,
  'Spotify': True,
  'Soundcloud': True,
  'Tidal': True}}

In [1]:
mme        = masterManualEntries()
cbs        = convertByteString()
mam        = masterArtistMerger()
mma        = masterMultiArtistDB()
manc       = masterArtistNameCorrection()
manDB      = masterArtistNameDB("main")
multimanDB = masterArtistNameDB("multi")

NameError: name 'masterManualEntries' is not defined

In [ ]:
from masterUtils import artistIDAlbumOfTheYear
aid = artistIDAlbumOfTheYear()
from masterDBGate import masterDBGate
mdbGate = masterDBGate()
mdbGate.getDBs()

from masterUtils import *
aids = {}
for db in mdbGate.dbs:
    try:
        aid = eval("artistID{0}()".format(db))
    except:
        aid = eval("artistIDSelf()")
    aids[db] = aid
aids['RateYourMusic'].getArtistID

In [ ]:
[aid.getArtistID(x) for x in ['https://www.albumoftheyear.org/artist/100-el-guincho/',
 'https://www.albumoftheyear.org/artist/10000-ten/',
 'https://www.albumoftheyear.org/artist/100200-harvey-matusows-jews-harp-band/',
 'https://www.albumoftheyear.org/artist/100400-small-bills/',
 'https://www.albumoftheyear.org/artist/100500']]

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

In [ ]:
masterArtistKey("HI")

In [3]:
from pandas import isna

class masterArtistKey(str):
    def __init__(self, key):
        if isinstance(key,str):
            self.key = key
        elif isinstance(key, masterArtistKey):
            self.key = str(key)
        elif isinstance(key, int):
            self.key = str(key)
        elif isna(key):
            raise ValueError("Key must be a str, int, or self. You gave a 'None'")
        else:
            raise ValueError("Key [{0}] must be a str, int, or self".format(key))        

In [ ]:
from pandas import Timestamp

class masterArtistTimestamp:
    def __init__(self):
        self.created  = Timestamp.today().round('min')
        self.modified = Timestamp.today().round('min')
        
    def update(self):
        self.modified = Timestamp.today().round('min')

In [ ]:
from masterDBGate import masterDBGate
from pandas import Series, DataFrame
from uuid import uuid4

class masterArtistDB:
    def __init__(self):
        self.db = Series(dtype = 'object')
        
    def add(self, ma):
        row = Series({ma.id: ma})
        self.db = self.db.append(row, verify_integrity=True)
        
    def N(self):
        return len(self.db)
        
    #def getData(self):
        
        

class masterArtist:
    def __init__(self):
        
        ### DB Entry Information
        self.timestamp = masterArtistTimestamp()
        self.id = uuid4()
        
        ### Artist Information
        self.name = None
        
        ### Artist DB Matches
        self.dbIDs = Series({db: masterArtistKey(None) for db in masterDBGate().getDBs()})
        
        
    def setName(self, name):
        self.name = name
        #self.timestamp.update()
        
        
    def setdbid(self, db, dbID):
        if isinstance(self.dbIDs.get(db), masterArtistKey):
            self.dbIDs[db] = masterArtistKey(dbID)
            
        
    def get()

In [ ]:
mmeDF = meu.getDF()

In [ ]:
ts = timestat("Creating MasterArtistDB")
maDB = masterArtistDB()
for idx,row in mmeDF[["ArtistName"]+masterDBGate().getDBs()].head(10000).iterrows():
    ma = masterArtist()
    for key,val in row.iteritems():
        if key == "ArtistName":
            ma.setName(val)
        else:
            ma.setdbid(key,val)
    
    maDB.add(ma)
ts.stop()

In [ ]:
ts = timestat("Creating DataFrame For {0} Entries".format(maDB.N()))
nameDF = DataFrame(Series({key: val.name for key,val in maDB.db.iteritems()}, name="ArtistName"))
ts.update()
dbidDF = DataFrame({key: val.dbIDs for key,val in maDB.db.iteritems()}).T
ts.update()
df     = nameDF.join(dbidDF)
ts.stop()

In [ ]:
df

In [ ]:
df = mme.getDataFrame()
df = DataFrame({col: colData.apply(lambda x: str(x) if notna(x) else None) for col,colData in df.iteritems()})

In [ ]:
ts = timestat("Getting ArtistID -> Clean Name Map")
mDiscs = masterDBGate().getDiscs()
artistIDToCleanName = {db: disc.getArtistIDToPreMergeNameData().apply(lambda x: manc.realName(x)[0]).apply(manc.clean).apply(cbs.convert) for db,disc in mDiscs.items()}
ts.stop()

In [ ]:
ts = timestat("Getting MergerID -> Name Map")
mergerIDToName = {db: {} for db in mDiscs.keys()}
for artistName,artistData in mam.getData().iteritems():
    for db,dbData in artistData.items():
        mergerIDToName[db][dbData["ID"]] = artistName
ts.stop()

In [ ]:
def getCleanArtistName(dbID, db):
    if isinstance(dbID,str):
        mergerName = mergerIDToName[db].get(dbID)
        if mergerName is not None:
            return (mergerName,dbID,True)
        
        cleanName = artistIDToCleanName[db].get(dbID)
        if cleanName is not None:
            return (cleanName,dbID,False)
        
        if not dbID.isdigit():
            return ("NotDigit",dbID,False)
        else:
            return ("NotInDB",dbID,False)
    elif isna(dbID):
        return None
    else:
        raise ValueError("Unsure how to get name for ID [{0}]/[{1}]".format(db,dbID))

ts = timestat("Joining ID To Name For {0} Entries And {1} DBs".format(df.shape[0],df.shape[1]))
dfNameData = DataFrame({db: dbDFData.apply(getCleanArtistName, db=db) for db,dbDFData in df.iteritems() if db in mDiscs})
colnames   = ["ArtistName"] + list(dfNameData.columns)
dfNameData = dfNameData.join(df["ArtistName"])[colnames]
ts.stop()

### Fix Merger IDs

In [ ]:
def fixMergerIDs(df, mam):
    dbMaxLen   = {db: df[db].apply(lambda x: len(x) if x is not None else 0).max() for db in artistIDToCleanName}
    mergedRows = concat([dbData[dbData.apply(lambda x: len(x) if x is not None else 0) == dbMaxLen[db]] for db,dbData in df.iteritems() if db in artistIDToCleanName]).index.drop_duplicates()


    idxs = []
    for idx,row in df.loc[mergedRows].iterrows():
        mergeData = mam.getArtistDataByName(row["ArtistName"])
        if mergeData is None:
            print(row["ArtistName"])
            idxs.append(idx)
            continue
        print(row["ArtistName"])
        for db,dbMergeData in mergeData.items():
            mergeID   = dbMergeData["ID"]
            currentID = row[db]
            print("\t{0: <16}{1}  -->  {2}".format(db,currentID,mergeID))
            df.loc[idx,db] = mergeID

In [ ]:
#mme.saveData(manualEntries=df, local=False)

In [ ]:
def isMerger(row):
    return sum([mam.getArtistDataByMergerID(dbID) is not None for dbID in row.values]) > 0
ts = timestat("Find Merged Artist Data")
mergedArtists = df.apply(isMerger, axis=1)
mergedIDXs    = df[mergedArtists].index
ts.stop()

In [ ]:
dfNameData[dfNameData["ArtistName"] == "Alice Cooper"]

In [ ]:
class artistGroup:
    def __init__(self, key, debug=False):
        self.key   = key
        self.debug = debug
        
        ############################################################################
        # General And Diagnostic
        ############################################################################
        self.groupType  = None
        self.terminal   = True # Becomes False If adding an artistGroup To groups()
        self.mmeID      = None
        
        
        ############################################################################
        # Database Matches
        ############################################################################
        self.dbIDs = {}
        
        
        ############################################################################
        # Artist Group Names
        ############################################################################
        
        ### Will likely be an ALL CAPS version of the assigned name
        self.searchName = None
        
        ### My Choice of Group Name (very arbitrary. must be in stylized or latin names)
        self.assignedName = None
        
        ### Stylized Names (any weird way group's name is written)
        self.stylizedNames = []
        
        ### Latin Names (Ascii if possible, something readable in English)
        self.latinNames = []
        
        ### Renames (Mapping between name and one of names in stylized or latin names)
        self.dbRenames  = {}
        self.genRenames = {}
        
        ### A collection of other ArtistGroup items
        self.groups = {}
        
        
    
    ################################################################################################################################
    # General
    ################################################################################################################################
    def show(self):
        print("{0: <20}: {1}".format("Key", self.key))
        print("{0: <20}: {1}".format("Assigned Name", self.assignedName))
        print("{0: <20}: {1}".format("Search Name", self.searchName))
        print("{0: <20}: {1}".format("DB Matches", self.dbIDs))
        print("{0: <20}: {1}".format("DB Renames", self.dbRenames))
        print("{0: <20}: {1}".format("General Renames", self.genRenames))
        
        
    ################################################################################################################################
    # Getters and Setters
    ################################################################################################################################
    def getKey(self):
        return self.key
    
    def setDBIDs(self, dbIDs):
        self.dbIDs = dbIDs
    
    def setAssignedName(self, assignedName):
        self.assignedName = assignedName
        self.searchName   = assignedName.upper()
        
    def setDBRenames(self, dbRenames):
        self.dbRenames = dbRenames
        
    def setGenRenames(self, genRenames):
        self.genRenames = genRenames
        
    def addGroup(self, ag):
        if isinstance(ag, artistGroup):
            self.groups[ag.getKey] = ag

In [ ]:
def createArtistGroupData(row, idx, manDB, mergedArtists):
    artistName = row["ArtistName"]
    
    artistDBData = {idx: idxData for idx,idxData in row.iteritems() if isinstance(idxData,tuple)}
    dbNames  = {db: dbData[0] for db,dbData in artistDBData.items() if dbData[0] not in ["NotInDB", "NotDigit"]}    
    dbIDs    = {db: dbData[1] for db,dbData in artistDBData.items()}
    isMerged = {db: dbData[2] for db,dbData in artistDBData.items() if dbData[2] is True}
    isMerged = isMerged if len(isMerged) > 0 else None
    if len(dbNames) == 0:
        print(idx,'\t',artistName)
    
    ag = artistGroup(key=key)
    ag.mmeID = idx
    ag.terminal = not isMerged
    ag.setAssignedName(artistName)

    unMerged = mergedArtists.isin([artistName]).sum() == 0
    if unMerged:
        dbRenames  = {db: {dbName: manDB.renamed(dbName)} for db,dbName in dbNames.items()}
        dbRenames  = {db: dbRename for db,dbRename in dbRenames.items() if list(dbRename.keys()) != list(dbRename.values())}
        genRenames = {rename: artistName for rename in manInvData.get(artistName, {}) if {rename: artistName} not in dbRenames.values()}
    else:
        dbRenames  = {}
        genRenames = {}
    ag.setDBRenames(dbRenames)
    ag.setGenRenames(genRenames)
    
    ag.setDBIDs(dbIDs)
    
    return ag

In [ ]:
indivAGS  = {}
mergedAGS = {}
N   = dfNameData.shape[0]
ts  = timestat("Creating Artist Groups For {0} \'Artists\'".format(N))
mergedArtists = df.loc[mergedIDXs]["ArtistName"]

for i,(idx,row) in enumerate(dfNameData.iterrows()):
    if (i+1) % 50000 == 0 or (i+1) == 10000:
        ts.update(n=i+1,N=N)
    
    key  = str(uuid4())
    data = createArtistGroupData(row, idx, manDB, mergedArtists)
    if idx in mergedIDXs:
        mergedAGS[key] = data
    else:
        indivAGS[key] = data
         
print("{0: <30}{1: >6}".format("All Artists", dfNameData.shape[0]))
print("{0: <30}{1: >6}".format("Individual Artists", len(indivAGS)))
print("{0: <30}{1: >6}".format("Merged Artists", len(mergedAGS)))

ts.stop()

In [ ]:
print("{0: <30}{1: >6}".format("All Artists", dfNameData.shape[0]))
print("{0: <30}{1: >6}".format("Individual Artists", len(indivAGS)))
print("{0: <30}{1: >6}".format("Merged Artists", len(mergedAGS)))

In [ ]:
ts = timestat("Split Renames By Known DB Renames")

manDBDataRemaining   = manDBData
ags = {"Individual": indivAGS, "Merged": mergedAGS}
for agType,agData in ags.items():
    dbRenameData = [item for item in getFlatList([ag.dbRenames.values() for key,ag in agData.items()]) if len(item) > 0]
    dbRenameData = {k: v for item in dbRenameData for k,v in item.items()}
    manDBDataTemp      = DataFrame(manDBDataRemaining, columns=["PermReplace"]).join(Series(dbRenameData, name="dbRename"))
    manDBDataRemaining = manDBDataTemp[manDBDataTemp["dbRename"].isna()]["PermReplace"]
    manDBDataDBRename  = manDBDataTemp[manDBDataTemp["dbRename"].notna()]["PermReplace"]

    print("{0: <30}{1: >6}".format("Perm Renames", manDBDataTemp.shape[0]))
    print("{0: <30}{1: >6}".format("Known DB Renames", manDBDataDBRename.shape[0]))
    print("{0: <30}{1: >6}".format("Remaining Renames", manDBDataRemaining.shape[0]))
ts.stop()



In [ ]:
ts = timestat("Split Renames By Known General Renames")
genRenameData = [ag.genRenames for key,ag in indivAGS.items() if len(ag.genRenames) > 0]
genRenameData = {k: v for item in genRenameData for k,v in item.items()}
manDBDataTemp      = DataFrame(manDBDataRemaining, columns=["PermReplace"]).join(Series(genRenameData, name="genRename"))
manDBDataRemaining = manDBDataTemp[manDBDataTemp["genRename"].isna()]["PermReplace"]
manDBDataGenRename = manDBDataTemp[manDBDataTemp["genRename"].notna()]["PermReplace"]

print("{0: <30}{1: >6}".format("(Perm-DB) Renames", manDBDataTemp.shape[0]))
print("{0: <30}{1: >6}".format("Known Gen Renames", manDBDataGenRename.shape[0]))
print("{0: <30}{1: >6}".format("Remaining Renames", manDBDataRemaining.shape[0]))
ts.stop()

In [ ]:
ts = timestat("Split Renames By Merged Renames")
manDBDataTemp        = manDBDataRemaining
manDBDataMergeRename = manDBDataTemp[manDBDataTemp.isin(df.loc[mergedIDXs]["ArtistName"])]
manDBDataRemaining   = manDBDataTemp[~manDBDataTemp.isin(df.loc[mergedIDXs]["ArtistName"])]
ts.stop()

print("{0: <30}{1: >6}".format("(Perm-DB-Merge) Renames", manDBDataTemp.shape[0]))
print("{0: <30}{1: >6}".format("Known Merge Renames", manDBDataMergeRename.shape[0]))
print("{0: <30}{1: >6}".format("Not Merge Renames", manDBDataRemaining.shape[0]))

In [ ]:
manDBDataRemaining[manDBDataRemaining.isin(["Dave Matthews"])]

In [ ]:
manDBDataMergeRename